# Agent高级规划（Planning）、记忆系统（Memory）与Tool Call
摆脱对上层框架封装的盲目依赖，深入到 OpenAI API 底层协议与原生算法细节，彻底明白 Agent 的“大脑”与“手脚”是如何进行通信的。

## Tool Calling 原生 Protocol 与 JSON Schema
1. 破除封装迷思：理解 LangChain / LlamaIndex 的 `@tool` 装饰器背后，到底向 OpenAI / Anthropic 等 API 发送了怎样的 JSON Schema。
2. 掌握 Tool Calling 的底层交互协议：深入理解 `tools` 参数定义、`tool_calls` 的响应结构，以及 Tool 返回结果后的 `tool` role 消息回传机制。
3. 原生 Python 实现无框架 Tool Calling 循环：不借助 LangChain/LangGraph，仅使用原生 `openai` SDK 手写一个完整的 ReAct/Tool-Calling 循环。

## LLM Tool Calling 底层通信协议


#### 为什么大模型能“调用工具”？
大模型本身并不能直接运行外部代码或 API。所谓的 Tool Calling，本质上是 LLM 按照约定的 JSON 结构输出特定文本，并由客户端代码解析后在本地执行函数，最后将执行结果再喂回 LLM 的闭环过程。

交互全流程分为 4 个阶段：
1. 定义与申明（Declaration）：开发者在请求中向 LLM 发送 tools 参数，内部包含符合 JSON Schema 规范的函数名、参数描述。
2. 模型决策（Decision / Tool Call）：LLM 判定需要调用函数，返回 `finish_reason: "tool_calls"` 并在消息中附带 `tool_calls`（含 `id`, `function.name`, `function.arguments` JSON 字符串）。
3. 客户端本地执行（Execution）：开发者解析 `tool_calls`，在 Python 本地调用实际函数并拿到返回值（如 API 响应或数据库结果）。
4. 结果回填与最终生成（Observation & Final Generation）：开发者构造一条 `role: "tool"` 的消息（带上对应 `tool_call_id`），把结果发送给 LLM，LLM 据此生成人类可读的最终回答。


#### JSON Schema 的重要性
LLM 能否准确识别参数类型，全赖于 `parameters` 字典中定义的 JSON Schema。例如：

In [ ]:
{
  "type": "function",
  "function": {
    "name": "get_weather",
    "description": "查询指定城市的天气预报",
    "parameters": {
      "type": "object",
      "properties": {
        "city": {
          "type": "string",
          "description": "城市名称，例如：北京、上海"
        },
        "unit": {
          "type": "string",
          "enum": ["celsius", "fahrenheit"],
          "description": "温度单位"
        }
      },
      "required": ["city"]
    }
  }
}

下面的代码完全不用任何 LangChain / LangGraph 封装，用原生 OpenAI SDK展示底层是如何手写处理 Tool Call 的：

In [ ]:
import json
import os
from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# -------------------------------------------------------------
# 1. 定义本地 Python 函数 (实际业务逻辑)
# -------------------------------------------------------------
def get_stock_price(symbol: str) -> str:
    """模拟查询股票价格"""
    prices = {"AAPL": "185.50 USD", "NVDA": "120.25 USD", "TSLA": "220.00 USD"}
    price = prices.get(symbol.upper(), "未知股票代码")
    return json.dumps({"symbol": symbol, "price": price})

def execute_sql_query(query: str) -> str:
    """模拟数据库查询"""
    return json.dumps({"status": "success", "rows_affected": 1, "data": [["User_A", "Active"]]})

# 本地函数映射表
available_tools = {
    "get_stock_price": get_stock_price,
    "execute_sql_query": execute_sql_query,
}

# -------------------------------------------------------------
# 2. 构造发送给 OpenAI API 的 Native JSON Schema 声明
# -------------------------------------------------------------
tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "get_stock_price",
            "description": "获取指定美股代码的最新实时股价",
            "parameters": {
                "type": "object",
                "properties": {
                    "symbol": {
                        "type": "string",
                        "description": "股票代码，如 AAPL, NVDA, TSLA"
                    }
                },
                "required": ["symbol"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "execute_sql_query",
            "description": "在数据库中执行只读 SQL 查询语句",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "合法的 SQL 查询字符串"
                    }
                },
                "required": ["query"]
            }
        }
    }
]

# -------------------------------------------------------------
# 3. 原生 Tool-Calling 交互主循环
# -------------------------------------------------------------
def run_native_agent(user_prompt: str):
    print(f"👤 用户提问: {user_prompt}\n")

    messages = [
        {"role": "system", "content": "你是一个严谨的助手。如果需要外部数据，请使用工具。"},
        {"role": "user", "content": user_prompt}
    ]

    # 第一次 API 调用：携带 tools 申明
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=tools_schema,
        tool_choice="auto"  # 让模型自主选择是否调用工具
    )

    response_message = response.choices[0].message
    messages.append(response_message) # 将大模型的回复加入上下文历史

    # 检查大模型是否提出了 tool_calls 申请
    if response_message.tool_calls:
        print("🤖 [LLM 决策]: 触发了 Tool Calling，准备执行以下本地函数：")

        for tool_call in response_message.tool_calls:
            function_name = tool_call.function.name
            function_args = json.loads(tool_call.function.arguments)
            tool_id = tool_call.id

            print(f" ⚙️ 函数名: {function_name}")
            print(f" 📦 解析得到的参数: {function_args}")
            print(f" 🔑 Tool Call ID: {tool_id}")

            # 找到对应的本地 Python 函数并执行
            function_to_call = available_tools[function_name]
            function_response = function_to_call(**function_args)

            print(f" 💡 工具执行返回结果: {function_response}\n")

            # **极其重要**：将工具执行结果作为 role: "tool" 消息回传
            messages.append({
                "tool_call_id": tool_id,
                "role": "tool",
                "name": function_name,
                "content": function_response,
            })

        # 第二次 API 调用：带着包含 tool 结果的完整上下文，让 LLM 输出最终回答
        second_response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
        )
        final_answer = second_response.choices[0].message.content
        print(f"🏁 [最终回答]: {final_answer}")
    else:
        print(f"🏁 [直接回答]: {response_message.content}")

if __name__ == "__main__":
    run_native_agent("请帮我查一下英伟达 (NVDA) 现在的股价是多少？")

1. 并发 Tool Calling（Parallel Function Calling）处理：
    * 在最新的 GPT-4o / Claude 3.5 中，如果用户问：“请同时查一下 AAPL 和 TSLA 的股价”，LLM 在一次响应中会返回包含多个 Tool Call 的 `tool_calls` 数组
    * 思考：在代码逻辑中，为什么必须将所有 Tool Call 的结果按照各自对应的 `tool_call_id` 挨个拼接到 `messages` 中？如果遗漏了某个 `tool_call_id` 会发生什么？

2. JSON Schema 校验与防爆注入
    * 大模型返回的 `tool_call.function.arguments` 是一个字符串（String），在使用 `json.loads()` 解析时，可能会遇到语法不合法或者缺少必要参数（`KeyError`）的情况。
    * 在生产级工程中，我们通常会配合 Python 的哪个类型校验库（提示：`Pydantic`）来对 LLM 传回的参数做 Safe Parsing？


## Agent 的规划能力（Planning）
一个成熟的 Agent 不能只依赖“走一步看一步”的盲目尝试，而是需要具备任务拆解（Task Decomposition）与自我反思（Self-Reflection）的能力。

1. 掌握三大 Agent 规划范式：
    * ReAct（Reasoning + Acting）：思考与行动交替交织。
    * Plan-and-Solve（先规划后执行）：先一次性生成全局子任务列表，再逐个执行。
    * Self-Ask（自问自答）：通过显式拆解中间子问题（Intermediate Questions）逐步逼近真相。

2. 理解反思与纠错机制（Self-Reflection / Reflexion）：当子步骤失败或工具报错时，Agent 如何评估反馈并修改后续计划。
3. 纯 Python 手写 Plan-and-Solve + Dynamic Replanning 架构：不依赖任何框架，用原生 Prompt 与 JSON 格式实现可动态重规划（Replanning）的 Agent 引擎。

#### Agent 三大规划范式对比
| 规划范式           | 核心机制                                                     | 优势                               | 局限性 / 适用场景                          |
|----------------|----------------------------------------------------------|----------------------------------|-------------------------------------|
| ReAct          | `Thought -> Action -> Observation -> Thought` 循环迭代       | 灵活适应动态环境，根据上一步 Observation 决定下一步 | 步骤较多时容易“迷路”或陷入死循环；Token 消耗较高        |
| Plan-and-Solve | `Plan (分解所有步骤) -> Execute (逐个执行) -> Replanning`          | 具备全局视角，适合复杂、多步骤的长流程任务            | 若初始 Plan 偏离路线，需要显示引入 Replanner 节点修正 |
| Self-Ask       | `Follow-up Query -> Intermediate Answer -> Final Answer` | 逻辑推导链极强，适合多跳问答（Multi-hop QA）     | 主要是知识检索与推理，缺乏对复杂环境的改变能力（Action）     |


Python 实现 Plan-and-Solve + 动态重规划 (Replanner)
1. Planner：输入用户大目标，输出 JSON 格式的子任务列表列表。
2. Executor：逐个执行子任务（调用工具或进行推理）。
3. Replanner：每完成一步或遇到异常，重新评估剩余步骤是否需要修改。


In [ ]:
import json
import os
from typing import List, Dict, Any
from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# -------------------------------------------------------------
# 1. 模拟工具库 (Tools)
# -------------------------------------------------------------
def search_web(query: str) -> str:
    """搜索网络信息"""
    query_lower = query.lower()
    if "天气" in query or "weather" in query_lower:
        return "北京今日天气：晴朗，18°C ~ 28°C。"
    elif "机票" in query or "flight" in query_lower:
        return "北京到上海航班：MU5102 次 10:00 起飞，票价 650 元。"
    return f"搜索 [{query}] 的结果：暂无异常信息。"

tools_map = {"search_web": search_web}

# -------------------------------------------------------------
# 2. Planner 节点：将大任务分解为 Task List
# -------------------------------------------------------------
PLANNER_SYSTEM_PROMPT = """你是一个高瞻远瞩的 AI 规划器 (Planner)。
你的任务是将用户的复杂请求拆解为一系列按顺序执行的子任务（Step-by-step Plan）。

你必须仅输出合法 JSON，格式如下：
{
  "steps": [
    "步骤1的描述",
    "步骤2的描述"
  ]
}
"""

def create_initial_plan(user_goal: str) -> List[str]:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": PLANNER_SYSTEM_PROMPT},
            {"role": "user", "content": f"目标：{user_goal}"}
        ],
        response_format={"type": "json_object"}
    )
    plan_data = json.loads(response.choices[0].message.content)
    return plan_data.get("steps", [])

# -------------------------------------------------------------
# 3. Replanner 节点：动态评估并更新 Plan
# -------------------------------------------------------------
REPLANNER_SYSTEM_PROMPT = """你是一个 AI 重规划器 (Replanner)。
基于用户的【原始目标】、目前【已完成的步骤及结果】，以及【剩余计划】：
请决定是继续执行原计划，还是修改剩余计划，或者宣布任务已完成。

你必须仅输出合法 JSON，格式如下：
{
  "is_completed": true/false,
  "final_response": "如果 is_completed 为 true，在此输出最终总结给用户",
  "remaining_steps": ["修改后的步骤1", "修改后的步骤2"] (如果 is_completed 为 false)
}
"""

def replan(user_goal: str, completed_history: List[str], current_remaining_steps: List[str]) -> Dict[str, Any]:
    history_str = "\n".join(completed_history)
    remaining_str = "\n".join(current_remaining_steps)

    prompt = f"""
    原始目标：{user_goal}
    已完成的步骤与执行结果：
    {history_str}

    当前剩余计划：
    {remaining_str}
    """

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": REPLANNER_SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)

# -------------------------------------------------------------
# 4. Executor 节点：单步子任务执行器
# -------------------------------------------------------------
EXECUTOR_SYSTEM_PROMPT = """你是一个 Agent 步骤执行器 (Executor)。
你需要执行给定的【单个子任务】。你可以直接回答，或者决定调用 search_web 工具。
如果你需要调用 search_web，请输出 JSON：{"action": "search_web", "query": "搜索关键词"}
如果不需要调用工具，直接输出 JSON：{"action": "final_answer", "answer": "结论"}
"""

def execute_step(step: str) -> str:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": EXECUTOR_SYSTEM_PROMPT},
            {"role": "user", "content": f"当前需执行的子任务：{step}"}
        ],
        response_format={"type": "json_object"}
    )
    res = json.loads(response.choices[0].message.content)

    if res.get("action") == "search_web":
        query = res.get("query")
        tool_result = search_web(query)
        return f"[调用 search_web({query})]: {tool_result}"
    else:
        return f"[思考结论]: {res.get('answer')}"

# -------------------------------------------------------------
# 5. Plan-and-Solve 主循环引擎
# -------------------------------------------------------------
def run_plan_and_solve(user_goal: str):
    print(f"🎯 任务目标: {user_goal}\n" + "="*50)

    # Step 1: 制定初始计划
    plan = create_initial_plan(user_goal)
    print("📋 [Planner] 生成的初始计划：")
    for i, step in enumerate(plan, 1):
        print(f"  {i}. {step}")
    print("="*50)

    completed_history = []
    remaining_steps = plan.copy()

    # Step 2: 动态循环执行与重规划
    step_count = 0
    while remaining_steps and step_count < 5:
        step_count += 1
        current_step = remaining_steps.pop(0)

        print(f"\n🚀 [Step {step_count} 执行中]: {current_step}")
        step_output = execute_step(current_step)
        print(f"   💡 执行反馈: {step_output}")

        # 记录执行历史
        completed_history.append(f"任务: {current_step} -> 结果: {step_output}")

        # 重新评估 Plan
        replan_res = replan(user_goal, completed_history, remaining_steps)

        if replan_res.get("is_completed"):
            print("\n🎉 [Replanner] 判定所有任务已达成！")
            print(f"🏁 最终答案: {replan_res.get('final_response')}")
            return
        else:
            remaining_steps = replan_res.get("remaining_steps", [])
            print(f"🔄 [Replanner] 动态更新后的剩余步骤: {remaining_steps}")

    print("\n⚠️ 达到最大步数限制或任务终止。")

if __name__ == "__main__":
    run_plan_and_solve("帮我查询明天北京的天气，并根据天气推荐一下去上海的机票费用。")

1. ReAct 与 Plan-and-Solve 的权衡：
    * 假设要开发一个“自动重构 10,000 行遗留 Python 代码”的智能体，你会优先选择 ReAct 还是 Plan-and-Solve + Replanner 架构？为什么？

2. 防止幻觉规划（Hallucinated Planning）：
    * 在 Planner 节点生成 Plan 时，如果它规划了一个本地根本没有对应工具 API 可以实现的步骤（例如：“步骤 3：直接发短信给用户手机”），我们在系统设计上该如何规避这种问题？


## Agent 记忆系统（Memory）
在实际生产环境中，大模型是“无状态”（Stateless）的，每一次 API 请求都是孤立的。如果不做记忆管理，对话一旦变长，不仅 Token 费用飙升，还会迅速超出 LLM 的上下文窗口（Context Window），甚至导致模型产生遗忘或混淆。所以需要短期记忆（Short-term Memory）管理与 Context Window 动态裁剪策略。

#### 短期对话记忆管理与 Context Window 动态裁剪策略
1. 理解 Agent 记忆分类：清晰区分短期记忆（Short-term Memory/Conversation Context）与长期记忆（Long-term Memory/Vector DB Store）。
2. 掌握三大短期记忆裁剪/压缩范式：
    * Sliding Window（滑动窗口 / Token 预算限制）：按 Token 数量截断，保留最新上下文。
    * Summary Memory（摘要记忆）：使用小模型/异步任务将历史对话实时提炼为 Summary。
    * Hybrid Memory（混合记忆：Summary + Recent Sliding Window）：固定系统提示 + 历史摘要 + 贴近当前的最新的 N 轮对话。

3. 纯 Python 手写轻量级内存管理引擎：实现一个具备 Token 预算控制与动态摘要压缩（Auto-Summarization）的 `MemoryManager` 类。

#### 短期记忆裁剪三大范式对比
| 记忆策略                  | 实现原理                                                       | 优点                              | 缺点 / 适用场景                          |
|-----------------------|------------------------------------------------------------|---------------------------------|------------------------------------|
| Sliding Window (滑动窗口) | 保留最近的 $K$ 条消息或控制总 Token 数不超过 `max_tokens`                  | 实现极简、计算开销小，响应速度极快               | 丢失早期的关键背景信息（如用户最开始提到的姓名、偏好）        |
| Summary Memory (摘要总结) | 随着对话推进，定期让 LLM 将旧消息归纳为一段简短摘要（Summary）                      | 占用 Token 极少，能长久保留关键事实           | 丢失对话细节与精准语气，频繁调用 LLM 产生额外延时与 Token |
| Hybrid Memory (混合策略)  | `System Prompt + Global Summary + Sliding Window Messages` | 工业界最常用！既保留近期对话的上下文连贯，又不遗忘早期重要事实 | 需要额外的逻辑维护摘要的更新时机                   |


下面的代码展示了如何在原生 Python 中实现一个工业级的 Hybrid Memory 管理器。当消息队列的总 Token 超过设定阈值时，自动提取旧消息进行增量摘要更新。

In [ ]:
import os
import json
from typing import List, Dict, Any
from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

class HybridMemoryManager:
    """
    混合动态记忆管理器
    - 维持 System Prompt
    - 维持增量对话摘要 (Summary)
    - 维持滑动窗口内的最新消息列表 (Recent Buffer)
    """
    def __init__(self, system_prompt: str, max_buffer_tokens: int = 500):
        self.system_prompt = system_prompt
        self.summary = ""  # 历史摘要
        self.buffer: List[Dict[str, str]] = []  # 近期消息滑动窗口
        self.max_buffer_tokens = max_buffer_tokens

    def _estimate_tokens(self, text: str) -> int:
        """简易 Token 估算：粗略按 字符数 / 3 或 词数计算（生产环境推荐 tiktoken）"""
        return len(text) // 2

    def _get_buffer_tokens(self) -> int:
        total = 0
        for msg in self.buffer:
            total += self._estimate_tokens(msg.get("content", ""))
        return total

    def add_message(self, role: str, content: str):
        """添加一条新消息（User / Assistant / Tool）"""
        self.buffer.append({"role": role, "content": content})
        # 检查是否超出了 Buffer 的 Token 阈值，若超出则触发压缩
        if self._get_buffer_tokens() > self.max_buffer_tokens:
            self._compress_memory()

    def _compress_memory(self):
        """将 Buffer 中较旧的后半部分消息弹出来，并增量合并进 Summary 中"""
        print("\n🧹 [Memory Engine]: 触发 Token 溢出保护，正在压缩旧记忆并更新 Summary...")

        # 提取 Buffer 前一半的消息进行摘要
        split_idx = len(self.buffer) // 2
        to_summarize = self.buffer[:split_idx]
        self.buffer = self.buffer[split_idx:]  # 留后一半作为滑动窗口

        # 构造 Summarizer Prompt
        history_to_compress = "\n".join([f"{m['role']}: {m['content']}" for m in to_summarize])

        prompt = f"""请将以下对话内容提取关键事实并补充更新到已有的摘要中。
已有摘要: {self.summary if self.summary else '无'}

新增需要压缩的对话:
{history_to_compress}

请输出更新后的简洁摘要（包含用户的关键信息、偏好以及发生的关键事件）："""

        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3
        )

        self.summary = response.choices[0].message.content.strip()
        print(f"📌 [更新后的全局摘要]: {self.summary}\n")

    def get_messages_for_llm(self) -> List[Dict[str, str]]:
        """构建最终发送给 LLM 的 Messages 列表"""
        messages = [{"role": "system", "content": self.system_prompt}]

        # 如果存在历史摘要，以 System 附录或特质消息拼接到最前面
        if self.summary:
            summary_context = f"【历史对话摘要记忆】：\n{self.summary}"
            messages.append({"role": "system", "content": summary_context})

        # 拼接近期的滑动窗口对话
        messages.extend(self.buffer)
        return messages

# -------------------------------------------------------------
# 运行模拟测试
# -------------------------------------------------------------
if __name__ == "__main__":
    memory = HybridMemoryManager(
        system_prompt="你是一个贴心的智能助手。",
        max_buffer_tokens=150  # 调小 Token 限制以便测试自动触发压缩
    )

    # 模拟多轮对话
    dialogues = [
        ("user", "你好，我是张三，我是一名住在北京的 Python 工程师。"),
        ("assistant", "你好张三！很高兴认识你。有什么我可以帮你的吗？"),
        ("user", "我平时特别喜欢吃川菜，尤其是麻婆豆腐。"),
        ("assistant", "记下了！北京有很多正宗的川菜馆，麻婆豆腐确实是一道下饭神器。"),
        ("user", "你能帮我推荐一下适合我这个周末去逛的北京公园吗？"),
        ("assistant", "推荐颐和园或者景山公园，周末风景非常好！"),
        ("user", "对了我刚才说我叫什么名字来着？还有我喜欢吃什么？")
    ]

    for role, content in dialogues:
        print(f"🗣️ {role}: {content}")
        memory.add_message(role, content)

    # 调用 LLM 回答最后一个问题
    print("="*50)
    print("🤖 发送给 LLM 的最终 Context 结构：")
    final_messages = memory.get_messages_for_llm()
    for m in final_messages:
        print(f"  [{m['role'].upper()}]: {m['content']}")
    print("="*50)

    final_res = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=final_messages
    )
    print(f"🏁 [LLM 最终回答]:\n{final_res.choices[0].message.content}")

1. Tool Call 消息在 Memory 裁剪中的特殊性：
    * 在涉及 Tool Calling 的对话历史中，如果滑动窗口裁剪时只裁掉了 `assistant` 的 `tool_calls` 消息，但保留了后面的 `role: "tool"` 消息，发送给 OpenAI API 时会发生什么？在工程上应该如何避免？

2. 异步摘要（Async Summarization）优化：
    * 在上面的实战代码中，`_compress_memory()` 是同步阻塞在用户请求主流程中的，这会导致用户那一步的响应等待时间变长。在高性能 Agent 系统中，我们应该如何设计异步记忆压缩架构？


## Agent长期记忆（Long-term Memory）
长期记忆（Long-term Memory）- 基于向量数据库/图数据库的存储机制与 User Profile（用户画像/偏好）

长期记忆三大存储形态：
* Semantic Memory（语义记忆/向量存储）：通过 Embedding + 向量数据库（Vector DB）存储历史知识与经验 。
* Episodic Memory（情景记忆/事件序列）：按时间线/事件链记录 Agent 过去的具体行为与执行结果。
* User Profile / Entity Memory（用户画像/实体偏好）：提取结构化的用户属性、行为偏好与习惯（JSON / KV Store / Knowledge Graph）。

#### Agent 短期记忆 vs 长期记忆
在构建复杂的 Agent 系统时，仅靠上下文窗口中的短期记忆是不够的。下表对比了两者在工程实现上的核心差异：

| 维度   | 短期记忆 (Short-term Memory)                | 长期记忆 (Long-term Memory)                      |
|------|-----------------------------------------|----------------------------------------------|
| 存储载体 | Context Window / LLM Prompt 缓冲区         | 向量数据库（Milvus / Chroma / Pinecone）、KV 存储、图数据库 |
| 生命周期 | 单次 Session 或几轮对话以内                      | 跨 Session、永久保存（Persisted）                    |
| 容量限制 | 受限于 LLM 的 Context Window（如 128k Tokens） | 理论上无上限                                       |
| 检索方式 | 直接读取 Sequence，先进先出或滑动窗口截断               | 语义相似度搜索（Vector Cosine Search）、实体关系图检索        |
| 核心应用 | 维持当前 Task 的逻辑连贯与 Tool Calling 状态        | 记住用户的长期偏好、性格特征以及跨天的历史执行经验                    |



#### User Profile（用户画像）的动态提取与沉淀
长期记忆中最关键的一环是 User Profile 抽取。其基本流程如下：
1. 观察（Observation）：分析用户最新的对话或指令。
2. 提取（Extraction）：利用小模型/提示词判断对话中是否包含用户的持久性事实（如：“我不吃辣”、“我是 Python 程序员”）
3. 合并/更新（Merge & Resolve Conflict）：将提取出的新标签与已有的 Profile JSON 相比对，覆盖冲突项并追加新事实。
4. 注入（Injection）：在未来的任何新 Session 中，将 User Profile 自动注入到 Agent 的 `System Prompt` 中。


下面的完整示例展示了如何将 结构化 User Profile 沉淀 与 语义向量检索（Semantic Long-term Memory） 整合到一个长期记忆管理器中：

In [ ]:
import os
import json
from typing import List, Dict, Any
from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

class LongTermMemoryManager:
    """
    Agent 长期记忆管理器：
    1. User Profile: 结构化保存用户画像与偏好 (JSON 格式)
    2. Episodic/Semantic Memory: 存储历史事件/对话的向量索引，支持 Cosine 相似度检索
    """
    def __init__(self, profile_path: str = "user_profile.json"):
        self.profile_path = profile_path
        self.user_profile: Dict[str, Any] = self._load_profile()
        # 内存中简易的向量数据库 (存储格式: [{"embedding": [...], "text": "..."}])
        self.vector_store: List[Dict[str, Any]] = []

    def _load_profile(self) -> Dict[str, Any]:
        if os.path.exists(self.profile_path):
            with open(self.profile_path, "r", encoding="utf-8") as f:
                return json.load(f)
        return {"name": "Unknown", "occupation": "Unknown", "preferences": []}

    def _save_profile(self):
        with open(self.profile_path, "w", encoding="utf-8") as f:
            json.dump(self.user_profile, f, ensure_ascii=False, indent=2)

    def extract_and_update_profile(self, user_input: str):
        """利用 LLM 从用户的发言中提取固定的个人事实/偏好并更新 Profile"""
        prompt = f"""分析以下用户的发言，判断其中是否包含用户的持久个人信息（如姓名、职业、居住地、喜好、禁忌等）。
已有 User Profile:
{json.dumps(self.user_profile, ensure_ascii=False)}

用户最新发言:
"{user_input}"

如果包含新信息或需要修改旧信息，请输出更新后的完整 JSON；如果不包含任何长期 Profile 信息，请直接原样输出原 JSON。
只返回纯 JSON，不要带 markdown 代码块或其他字符："""

        res = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0
        )
        try:
            updated_profile = json.loads(res.choices[0].message.content.strip())
            if updated_profile != self.user_profile:
                self.user_profile = updated_profile
                self._save_profile()
                print(f"👤 [User Profile 动态更新成功]: {self.user_profile}")
        except json.JSONDecodeError:
            pass

    def add_to_long_term_memory(self, text: str):
        """将重要历史片段存入长期向量记忆库"""
        emb_res = client.embeddings.create(
            input=text,
            model="text-embedding-3-small"
        )
        embedding = emb_res.data[0].embedding
        self.vector_store.append({"embedding": embedding, "text": text})
        print(f"🧠 [长期记忆索引构建完成]: \"{text}\"")

    def recall_memory(self, query: str, top_k: int = 2) -> List[str]:
        """根据 Query 计算余弦相似度，检索最相关的长期记忆"""
        if not self.vector_store:
            return []

        q_emb = client.embeddings.create(
            input=query,
            model="text-embedding-3-small"
        ).data[0].embedding

        def cosine_similarity(v1, v2):
            dot = sum(a * b for a, b in zip(v1, v2))
            norm_v1 = sum(a * a for a in v1) ** 0.5
            norm_v2 = sum(b * b for b in v2) ** 0.5
            return dot / (norm_v1 * norm_v2) if norm_v1 and norm_v2 else 0.0

        scored_memories = [
            (cosine_similarity(q_emb, item["embedding"]), item["text"])
            for item in self.vector_store
        ]
        # 按相似度降序排列
        scored_memories.sort(key=lambda x: x[0], reverse=True)
        return [mem[1] for mem in scored_memories[:top_k] if mem[0] > 0.3]


# -------------------------------------------------------------
# 运行模拟测试
# -------------------------------------------------------------
if __name__ == "__main__":
    lt_memory = LongTermMemoryManager()

    # 1. 模拟历史会话（几天前）存入的长期经验与用户偏好
    print("--- Phase 1: 历史记忆沉淀与 Profile 抽取 ---")
    lt_memory.extract_and_update_profile("你好！我是李雷，我目前在一家自动驾驶公司做算法工程师。")
    lt_memory.extract_and_update_profile("我对海鲜严重过敏，千万不能吃虾和蟹。")

    # 存入一些过去任务的经验/知识片段
    lt_memory.add_to_long_term_memory("用户曾在2025年11月做过一次关于 PyTorch C++ 扩展编译报错的排查，解决方法是检查 CUDA 版本匹配。")
    lt_memory.add_to_long_term_memory("用户偏好使用 Clean Code 风格的 Python 3.11 代码，并且习惯附带完整的 Type Hints。")

    print("\n--- Phase 2: 新会话中的 Profile 注入与长期记忆召回 ---")
    current_query = "你能给我写一段处理传感数据的 Python 脚本吗？另外顺便帮我点一份晚餐建议。"

    # 提取 Profile 事实 + 检索长期知识
    recalled_mems = lt_memory.recall_memory(current_query)

    # 构造发送给大模型的上下文
    system_prompt = f"""你是一个智能助手。
【用户基本画像 (Profile)】:
{json.dumps(lt_memory.user_profile, ensure_ascii=False)}

【召回的长期历史记忆】:
{chr(10).join(['- ' + m for m in recalled_mems])}
"""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": current_query}
    ]

    print("🤖 发送给 LLM 的上下文 (包含 Profile & Recalled Memories):")
    print(system_prompt)

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        temperature=0.7
    )

    print("🏁 [LLM 最终回答]:")
    print(response.choices[0].message.content)


1. Profile 冲突与过期更新问题：

   * 如果用户在几个月前说“我住在北京”，今天突然说“我上周搬到了上海”，系统应该如何设计更新逻辑，以防 User Profile 中出现逻辑矛盾（如既写住在北京又写住在上海）？
2. 长期记忆检索的时效性衰减（Memory Decay）：

    * 在纯向量检索中，如果几年前的记忆与今天的记忆在语义上非常接近，向量相似度可能完全相同。在实际工程落地时，我们应该引入什么公式或权重机制（如依据时间戳的 Recency Score）来平衡 语义相关性 与 记忆新鲜度？


#### 整合Agent核心组件
1. Memory Management：结合滑动窗口 + 增量摘要（短期记忆）以及 User Profile JSON 持久化（长期记忆）。
2. Planning & Re-planning：在复杂任务下拆解 Task，并基于工具执行的 Observation 实时更新剩余 Task。
3. Action / Tool Execution：标准化 JSON Function Schema 的匹配与本地函数反射调用。

构建鲁棒的 Agent Loop 主循环：处理工具调用失败、格式校验错误以及最大步数限制（Max Iteration Safeguard）。
Agent 架构流程图:


In [ ]:
[用户输入 Request]
       │
       ▼
[1. Profile & Long-term Memory 注入] ───► 读取/更新 user_profile.json
       │
       ▼
[2. Planner 节点] ─────────────► 生成初始 Task List
       │
       ├─────────────────────────────────────────┐
       ▼                                         │ Loop (循环执行)
[3. Step Executor 节点]                          │
       │                                         │
       ├──► [Tool Calling?] ──► 执行本地 API      │
       │         │                               │
       │         ▼                               │
       │    [Observation]                        │
       │         │                               │
       ▼         ▼                               │
[4. Memory Engine (短期/摘要)]                    │
       │                                         │
       ▼                                         │
[5. Replanner 节点] ──────────────────────────────┘
       │
       ├─► (未完成) ──► 更新 Task List, 继续下一轮
       └─► (已完成) ──► 结合全局上下文输出最终回答 -> 用户

以下是完全手写的生产级 Agent 引擎实现：

In [ ]:
import os
import json
from typing import List, Dict, Any, Callable
from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# =============================================================
# 1. 工具定义 (Tools)
# =============================================================
def get_user_calendar(date: str) -> str:
    """查询指定日期的日程安排"""
    if "明天" in date or "2026" in date:
        return "14:00 与架构师团队讨论 Agent 存储方案；18:00 健身。"
    return "全天无紧急日程。"

def search_restaurants(cuisine: str, location: str) -> str:
    """查询符合菜系和位置的餐厅"""
    return f"推荐[{location}]的[{cuisine}]餐厅：1. 蜀香阁 (人均120, 评分4.8)；2. 川味观 (人均90, 评分4.6)。"

TOOLS_SCHEMA = [
    {
        "name": "get_user_calendar",
        "description": "查询指定日期的日程安排",
        "parameters": {
            "type": "object",
            "properties": {"date": {"type": "string", "description": "日期，如 '明天' 或 '2026-08-07'"}},
            "required": ["date"]
        }
    },
    {
        "name": "search_restaurants",
        "description": "查询符合菜系和位置的餐厅",
        "parameters": {
            "type": "object",
            "properties": {
                "cuisine": {"type": "string", "description": "菜系，如 '川菜'"},
                "location": {"type": "string", "description": "地点，如 '北京海淀'"}
            },
            "required": ["cuisine", "location"]
        }
    }
]

TOOL_FUNCTIONS: Dict[str, Callable] = {
    "get_user_calendar": get_user_calendar,
    "search_restaurants": search_restaurants
}

# =============================================================
# 2. 记忆组件 (Memory Manager)
# =============================================================
class FullMemoryManager:
    def __init__(self, profile_file: str = "agent_profile.json"):
        self.profile_file = profile_file
        self.profile = self._load_profile()
        self.summary = ""
        self.buffer: List[Dict[str, Any]] = []

    def _load_profile( me ) -> Dict[str, Any]:
        if os.path.exists(self.profile_file):
            try:
                with open(self.profile_file, 'r', encoding='utf-8') as f:
                    return json.load(f)
            except Exception:
                pass
        return {"name": "张三", "location": "北京海淀", "dietary_preference": "喜欢吃川菜，不能吃羊肉"}

    def add_message(self, role: str, content: str, tool_calls=None, tool_call_id=None):
        msg = {"role": role, "content": content}
        if tool_calls:
            msg["tool_calls"] = tool_calls
        if tool_call_id:
            msg["tool_call_id"] = tool_call_id
        self.buffer.append(msg)

    def get_messages_for_llm(self, system_prompt: str) -> List[Dict[str, Any]]:
        messages = [{"role": "system", "content": system_prompt}]
        if self.profile:
            messages.append({"role": "system", "content": f"【用户持久特征 (Profile)】: {json.dumps(self.profile, ensure_ascii=False)}"})
        if self.summary:
            messages.append({"role": "system", "content": f"【历史对话摘要】: {self.summary}"})
        messages.extend(self.buffer)
        return messages

# =============================================================
# 3. 整合型 Agent 主引擎 (Full Agent Engine)
# =============================================================
class IntegratedAgentEngine:
    def __init__(self):
        self.memory = FullMemoryManager()

    def run(self, user_goal: str):
        print(f"🎯 用户目标: {user_goal}\n" + "="*60)
        self.memory.add_message("user", user_goal)

        # -----------------------------------------------------
        # Step 1: 阶段一 - 制定全局 Plan
        # -----------------------------------------------------
        planner_prompt = f"""你是一个智能 Agent 规划器。
请将用户的复杂目标分解为 2-4 个顺序子步骤。
必须仅输出 JSON：{{"steps": ["步骤1", "步骤2"]}}
"""
        plan_res = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "system", "content": planner_prompt}, {"role": "user", "content": user_goal}],
            response_format={"type": "json_object"}
        )
        steps = json.loads(plan_res.choices[0].message.content).get("steps", [])
        print("📋 [Planner] 生成初始计划：")
        for i, s in enumerate(steps, 1):
            print(f"  {i}. {s}")
        print("="*60)

        # -----------------------------------------------------
        # Step 2: 阶段二 - 循环执行 Task & 动态重规划
        # -----------------------------------------------------
        max_iterations = 6
        step_idx = 0

        while steps and step_idx < max_iterations:
            step_idx += 1
            current_step = steps.pop(0)
            print(f"\n🚀 [Iteration {step_idx}] 执行子任务: {current_step}")

            sys_prompt = f"""你是一个具备 Tool Calling 能力的执行 Agent。
当前整体目标：{user_goal}
当前需要处理的子任务：{current_step}
如果需要调用工具，请使用系统提供的 tools。请严格遵循逻辑顺序。"""

            messages = self.memory.get_messages_for_llm(sys_prompt)

            # 调用大模型执行当前 Step
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=messages,
                functions=TOOLS_SCHEMA,
                function_call="auto"
            )
            msg = response.choices[0].message

            # 校验是否有 Tool Call
            if msg.function_call:
                fn_name = msg.function_call.name
                fn_args = json.loads(msg.function_call.arguments)
                print(f"🛠️ [Tool Call]: 触发 API -> {fn_name}({fn_args})")

                # 执行本地函数
                tool_func = TOOL_FUNCTIONS.get(fn_name)
                if tool_func:
                    observation = tool_func(**fn_args)
                else:
                    observation = f"错误: 未找到工具 {fn_name}"

                print(f"💡 [Observation]: {observation}")

                # 将工具调用与观察结果记入 Memory
                self.memory.add_message("assistant", None, tool_calls=[{"id": f"call_{step_idx}", "type": "function", "function": {"name": fn_name, "arguments": msg.function_call.arguments}}])
                self.memory.add_message("function", observation, tool_call_id=f"call_{step_idx}")

            else:
                answer = msg.content
                print(f"💬 [Step 结论]: {answer}")
                self.memory.add_message("assistant", answer)

            # -----------------------------------------------------
            # Step 3: Replanner 评估剩余 Plan
            # -----------------------------------------------------
            replanner_prompt = f"""你是一个任务进度评估员。
用户总目标：{user_goal}
目前剩余计划：{json.dumps(steps, ensure_ascii=False)}

请评估当前是否已获取足够信息回答用户总目标？
输出 JSON 格式：
{{
  "is_finished": true/false,
  "updated_remaining_steps": ["剩余步骤1"] (若 is_finished 为 false)
}}"""

            replan_res = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "system", "content": replanner_prompt}],
                response_format={"type": "json_object"}
            )
            replan_data = json.loads(replan_res.choices[0].message.content)

            if replan_data.get("is_finished"):
                print("\n🎉 [Replanner] 判定所有必要信息已收集完毕，准备输出最终答案！")
                break
            else:
                steps = replan_data.get("updated_remaining_steps", steps)
                print(f"🔄 [Replanner] 更新剩余计划: {steps}")

        # -----------------------------------------------------
        # Step 4: 阶段三 - 生成最终全局回复
        # -----------------------------------------------------
        final_prompt = "请根据前面对话获取的所有信息与工具结果，综合性地给用户一个详尽、有条理的最终答复。"
        final_messages = self.memory.get_messages_for_llm(final_prompt)

        final_response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=final_messages
        )

        print("\n" + "="*60 + "\n🏁 [Agent 最终交付结果]:")
        print(final_response.choices[0].message.content)

# -------------------------------------------------------------
# 运行测试
# -------------------------------------------------------------
if __name__ == "__main__":
    engine = IntegratedAgentEngine()
    engine.run("帮我查一下我明天的日程安排，并根据我的饮食偏好推荐一家适合晚餐的餐厅。")

1. Tool Message 依赖性问题：在原生的 Tool Calling 流程中，OpenAI 等模型规定 `function / tool` 类型的消息必须紧跟在包含对应 `tool_calls` 的 `assistant` 消息之后。如果在 Memory 的滑动裁剪中，误将 `assistant(tool_calls)` 消息裁掉，却保留了 `tool` 消息，API 会直接抛出 `400 Invalid Message Sequence` 错误。你在自己的 Memory 引擎中会如何设计消息成对裁剪（Atomic Pair Trimming）机制来解决这个问题？
2. 死循环与兜底策略（Agent Infinite Loops）：如果 Tool 返回的 Observation 始终是报错信息，Replanner 节点可能会不断生成重复的子任务导致系统陷入无限循环。在工业级 Agent 框架（如 LangGraph/AutoGPT）中，通常采用哪些手段来做死循环检测与降级兜底？
